# Pretraining notebook

## Section 1: Generate text before training

Load your current model

Enter a few prompts and note the output.

Try these:
 - love is
 - baby
 - tonight

Generate 20 tokens and note the output.

Reflection: Is there any quality or meaning to the output or is it largely random text?  Why?

In [1]:
import torch

from weird_ai import config
from weird_ai.tokenizer import SimpleCharacterTokenizer
from weird_ai.generation import generate_and_print_sample

# This makes results more reproducible and reduces confusion when comparing outputs.
torch.manual_seed(123)

from weird_ai.model import WeirdAIModel

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(device)

# TODO:
# Load your model

with open(config.SAMPLE_LYRICS_FILE, "r", encoding="utf-8") as f:
    raw_text = f.read()

tokenizer = SimpleCharacterTokenizer(raw_text)

emb_dim = 64
context_length = 64
num_heads = 4
num_layers = 2

model = WeirdAIModel(
    vocab_size=len(tokenizer.chars),
    emb_dim=emb_dim,
    context_length=context_length,
    num_heads=num_heads,
    num_layers=num_layers
)
model.to(device)

# TODO:
# Generate text from:
# "love is"
# "baby"
# "tonight"
for prompt in ["love is", "baby", "tonight"]:
    generate_and_print_sample(model, tokenizer, device, prompt, context_length, max_new_tokens=20)

cuda
love is®우치ğ你世있想소베위결장оてUâ—Ñ잠
baby가Á져久LÓカ離会8想소베很ā뺏久色운제
tonight世爱금디悟陪장:개베wの베인てU폼к點ä


## Section 2: Understanding Logits

Using the logits starter code, do the following:
 - Apply softmax
 - Identify the highest probability token

Reflection: Why is softmax necessary before interpreting logits as probabilities?

In [2]:
logits = torch.tensor([
    [1.5, 2.0, 0.5]
])

probabilities = torch.softmax(logits, dim=-1)

print(probabilities)
print(probabilities.sum())

predicted_token = torch.argmax(probabilities)

print(predicted_token)

tensor([[0.3315, 0.5465, 0.1220]])
tensor(1.0000)
tensor(1)


## Section 3: Cross Entropy Loss

### Textbook 
The textbook discussion begins on page 136

### Manually calculate
 - Probabilities
 - Log probabilities
 - Average negative log probabilities

### Compare with torch
Compare your manual results to the torch cross entropy results

```python
     torch.nn.functional.cross_entropy(...)
'''


In [3]:
import torch

# Manually calculate values

probs = torch.tensor([
    0.7,
    0.2,
    0.1
])

target_index = 0

In [4]:
target_probability = probs[target_index]

print(target_probability)

tensor(0.7000)


In [5]:
log_probability = torch.log(target_probability)

print(log_probability)

tensor(-0.3567)


In [6]:
loss = -log_probability

print(loss)

tensor(0.3567)


In [7]:
# Compare with torch calculations
logits = torch.log(probs).unsqueeze(0)
target = torch.tensor([target_index])

torch_loss = torch.nn.functional.cross_entropy(logits, target)
print(torch_loss)

tensor(0.3567)


## Section 4: Perplexity

Compute the perplexity

```python
perplexity = torch.exp(loss)
```

**Note:** A perplexity of 10 means the model is roughly as uncertain as choosing among 10 equally likely next tokens.

In [8]:
loss = torch.tensor(2.5)

perplexity = torch.exp(loss)

print(perplexity)

tensor(12.1825)


## Section 5: Understanding the Training Loop

The training loop performs:

1. Iterate through epochs
2. Iterate through batches
3. Zero gradients
4. Calculate loss
5. Backpropagation
6. Optimizer step
7. Evaluate model
8. Generate sample text

Draw or explain this process in your own words.

# Training vs. Validation

You will:
 - Split lyrics into train/validation
 - Create loaders
 - Compute initial loss

Keep note of your: 
 - Training Loss: 
 - Validation Loss: 

In [9]:
from weird_ai.dataset import LyricsDataset

train_ratio = 0.9

split_idx = int(train_ratio * len(raw_text))
train_text = raw_text[:split_idx]
val_text = raw_text[split_idx:]

print("Training characters:", len(train_text))
print("Validation characters:", len(val_text))

Training characters: 12156964
Validation characters: 1350774


In [10]:
# TODO:
# Create datasets
train_tokens = tokenizer.encode(train_text)
val_tokens = tokenizer.encode(val_text)

train_dataset = LyricsDataset(train_tokens, block_size=context_length)
val_dataset = LyricsDataset(val_tokens, block_size=context_length)

# TODO:
# Create dataloaders
from torch.utils.data import DataLoader

batch_size = 512

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

print(train_loader)
print(val_loader)

In [11]:
# TODO:
# Calculate initial losses

from weird_ai.losses import calc_loss_loader

with torch.no_grad():
    initial_train_loss = calc_loss_loader(train_loader, model, device, num_batches=35)
    initial_val_loss = calc_loss_loader(val_loader, model, device, num_batches=35)

print("Initial training loss:", initial_train_loss)
print("Initial validation loss:", initial_val_loss)

Initial training loss: 6.673345565795898
Initial validation loss: 6.667914390563965


## Section 6: Training

Train for one Epoch
```python
train_model(...)
```

Then record the loss before and after training.  

In [12]:
num_epochs = 1

# TODO:
# Train the model
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

from weird_ai.trainer import train_model_simple
train_losses, val_losses, tokens_seen = train_model_simple(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    device=device,
    num_epochs=num_epochs,
    eval_freq=200,
    eval_iter=10,
    start_context="love is",
    tokenizer=tokenizer,
    context_size=context_length
)

# TODO:
# Record:
# Training loss before
# Training loss after
initial_loss = initial_train_loss
final_loss = train_losses[-1]

Epoch 1, Step 0: train loss 6.641, val loss 6.633
Epoch 1, Step 200: train loss 2.972, val loss 2.991
Epoch 1, Step 400: train loss 2.600, val loss 2.652
Epoch 1, Step 600: train loss 0.219, val loss 0.253
Epoch 1, Step 800: train loss 0.087, val loss 0.109
Epoch 1, Step 1000: train loss 0.065, val loss 0.080
Epoch 1, Step 1200: train loss 0.055, val loss 0.066
Epoch 1, Step 1400: train loss 0.051, val loss 0.058
Epoch 1, Step 1600: train loss 0.048, val loss 0.053
Epoch 1, Step 1800: train loss 0.047, val loss 0.049
Epoch 1, Step 2000: train loss 0.044, val loss 0.046
Epoch 1, Step 2200: train loss 0.044, val loss 0.044
Epoch 1, Step 2400: train loss 0.041, val loss 0.042
Epoch 1, Step 2600: train loss 0.040, val loss 0.041
Epoch 1, Step 2800: train loss 0.039, val loss 0.040
Epoch 1, Step 3000: train loss 0.039, val loss 0.039
Epoch 1, Step 3200: train loss 0.037, val loss 0.038
Epoch 1, Step 3400: train loss 0.036, val loss 0.037
Epoch 1, Step 3600: train loss 0.036, val loss 0.037


In [13]:
print(f"Initial loss: {initial_loss}")
print(f"Final loss: {final_loss}")

Initial loss: 6.673345565795898
Final loss: 0.025248631834983826


In [14]:
import os
from weird_ai.trainer import save_checkpoint

os.makedirs("models/lesson-05-pretrained", exist_ok=True)

checkpoint_path = "models/lesson-05-pretrained/checkpoint.pt"

save_checkpoint(
    model=model,
    optimizer=optimizer,
    epoch=num_epochs,
    train_losses=train_losses,
    val_losses=val_losses,
    track_tokens_seen=tokens_seen,
    checkpoint_path=checkpoint_path
)

print("Checkpoint saved to", checkpoint_path)

Checkpoint saved to models/lesson-05-pretrained/checkpoint.pt


In [15]:
from weird_ai.trainer import load_checkpoint
from weird_ai.generation import generate_and_print_sample

restored_model = WeirdAIModel(
    vocab_size=len(tokenizer.chars),
    emb_dim=emb_dim,
    context_length=context_length,
    num_heads=num_heads,
    num_layers=num_layers
)
restored_model.to(device)
restored_optimizer = torch.optim.AdamW(restored_model.parameters(), lr=3e-4)

checkpoint_metadata = load_checkpoint(restored_model, restored_optimizer, checkpoint_path, device)

print("Restored epoch:", checkpoint_metadata["epoch"])
print("Restored train_losses:", checkpoint_metadata["train_losses"][-5:])
print("Restored val_losses:", checkpoint_metadata["val_losses"][-5:])
print("Restored tokens_seen:", checkpoint_metadata["track_tokens_seen"][-5:])

generate_and_print_sample(restored_model, tokenizer, device, "love is", context_length, max_new_tokens=20)

Restored epoch: 1
Restored train_losses: [0.025735734030604362, 0.026345698162913322, 0.026534168049693108, 0.02525302581489086, 0.025248631834983826]
Restored val_losses: [0.02822536788880825, 0.028315216302871704, 0.028132924810051918, 0.028004318475723267, 0.02832019329071045]
Restored tokens_seen: [747143168, 753696768, 760250368, 766803968, 773357568]
love issseediceo ccelele it


## Evaluation

Did the loss decrease?

Why is decreasing loss important?

## Before and After Comparison

Prompt:

love is

Before Training:
__________________

After Training:
__________________

Reflection:

How did the output change?

What evidence do you see that the model learned something from the training data?